In [1]:
#GLOBAL IMPORTS
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window



In [2]:
spark = SparkSession.builder \
    .appName("Final Assignment") \
    .master("local[*]") \
    .getOrCreate()


**PySpark Assignment**

**1)Basic DataFrame Operations**

1. Load the sales.csv and customer.csv files into separate DataFrames.

2. Display the schema of both DataFrames.

3. Show the first 5 rows from the sales
DataFrame.
4. Count the number of rows and columns in the customer DataFrame.

In [5]:
#Loading CSVs
sales_df=spark.read.csv("/content/sales.csv",inferSchema=True,header=True)
cust_df = spark.read.csv("/content/customers.csv",inferSchema=True,header=True)

#Displaying Schemas
sales_df.printSchema()
cust_df.printSchema()

#Display 5 Rows
sales_df.show(5)
cust_df.show(5)

#Counting Rows and Columns
sales_df.select(count("*").alias("Rows")).show(5)

cust_df.select(count("*").alias("Rows")).show(5)

print(f"Column in sales:{len(sales_df.columns)}")
print(f"Column in customer:{len(cust_df.columns)}")


root
 |-- sales_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- region: string (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)

+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|       1|       3842| Tablet| 77646|2025-09-07|  East|
|       2|       9089| Tablet| 57974|2024-04-27| South|
|       3|       3455|Desktop| 77588|2026-03-15| North|
|       4|       3951|Desktop|  9585|2024-06-28| South|
|       5|       8791| Mobile| 95976|2025-06-29|  West|
+--------+-----------+-------+------+----------+------+
only showing top 5 rows
+-----------+---------

**2. Data Cleaning Operations**

5. Remove duplicate rows from the sales DataFrame based on
customer_id,product,amount,sale_date,region columns
6. Drop rows where any column in the customer DataFrame has null values.
7. Replace null values in the amount column of the sales DataFrame with 0.
8. Replace null values in the email column of the customer DataFrame with the value
"unknown"

In [6]:
sales_df=sales_df.dropDuplicates(["customer_id","product","amount","sale_date","region"])

cust_df_drop=cust_df.na.drop()

sales_df=sales_df.fillna({"amount":0})

cust_df=cust_df.fillna({"email": "unknown"})

**3. Column Manipulation**

9. Add a new column discounted_amount to the sales DataFrame that applies a 10%
discount on amount.

10. Rename the city column in the customer DataFrame to customer_city.

11. Drop the region column from the sales DataFrame.

12. Create a new column customer_age_category in the customer DataFrame based on age:
a. "Youth" for age < 30
b. "Adult" for 30 <= age < 50
c. "Senior" for age >= 50


In [7]:
sales_df=sales_df.withColumn("discounted_amount",col("amount")*1.10)

cust_df = cust_df.withColumnRenamed("city","customer_city")

sales_df_drop=sales_df.drop("region")

cust_df = cust_df.withColumn("customer_age_category" ,when(col("age") <30 ,"youth") \
                              .when((col("age") >=30) & (col("age") <50),"adult")   \
                              .otherwise("senior"))


**4. Filtering**

13. Filter the sales DataFrame to show only rows where amount is greater than 50,000.

14. Filter the customer DataFrame to show customers aged between 25 and 30.

15. Identify all customers who have made purchases in more than one region.

16. Filter the top 3 sales based on amount for each product.

In [8]:
sales_df.filter("amount>50000").show(5)

cust_df.filter("age between 25 and 30").show(5)

cust_sales_df=sales_df.join(cust_df,sales_df.customer_id==cust_df.customer_id,"inner").drop(cust_df.customer_id)

cust_sales_df.groupBy("customer_id").agg(count_distinct("region").alias("region_count")) \
.filter("region_count>1").select("customer_id").show(5)


sales_df.groupBy("product").agg(max("amount").alias("max_amount")) \
.orderBy("max_amount",ascending=False).show(5)

+--------+-----------+-------+------+----------+------+------------------+
|sales_id|customer_id|product|amount| sale_date|region| discounted_amount|
+--------+-----------+-------+------+----------+------+------------------+
|     487|       7514| Tablet| 88686|2025-06-11|  West|           97554.6|
|     791|       6538|Desktop| 73428|2024-09-02|  East|           80770.8|
|     944|       6609| Mobile| 75109|2025-12-04| North| 82619.90000000001|
|    1642|       9605| Laptop| 84033|2024-07-12| North|           92436.3|
|    1861|       9586| Laptop| 96120|2025-05-20| North|105732.00000000001|
+--------+-----------+-------+------+----------+------+------------------+
only showing top 5 rows
+-----------+----------------+--------------------+---+-----------------+---------------------+
|customer_id|   customer_name|               email|age|    customer_city|customer_age_category|
+-----------+----------------+--------------------+---+-----------------+---------------------+
|        101|

**5. Joins**

17. Perform an inner join between sales and customer DataFrames on customer_id.

18. Perform a left join to include all records from sales and matching records from
customer.

19. Perform a full outer join between sales and customer DataFrames.

20. Identify customers who have not made any purchases by performing an anti-join.

In [10]:
join_df = sales_df.join(cust_df, sales_df.customer_id ==cust_df.customer_id,how="inner")

left_df= sales_df.join(cust_df,sales_df.customer_id == cust_df.customer_id ,how="left").show(5)

full_df=sales_df.join(cust_df, sales_df.customer_id == cust_df.customer_id ,how= "full").show(5)

anti_df = sales_df.join(cust_df , sales_df.customer_id == cust_df.customer_id , how = "left_anti").show(5)

+--------+-----------+-------+------+----------+------+------------------+-----------+---------------+--------------------+---+--------------------+---------------------+
|sales_id|customer_id|product|amount| sale_date|region| discounted_amount|customer_id|  customer_name|               email|age|       customer_city|customer_age_category|
+--------+-----------+-------+------+----------+------+------------------+-----------+---------------+--------------------+---+--------------------+---------------------+
|     339|        225| Mobile| 39862|2025-02-24| North|43848.200000000004|        225|Sherry Mckinney|mrussell@example.org| 42|          Laurieberg|                adult|
|     487|       7514| Tablet| 88686|2025-06-11|  West|           97554.6|       7514|  Tommy Wheeler|  jean08@example.com| 49|         Brandonberg|                adult|
|     791|       6538|Desktop| 73428|2024-09-02|  East|           80770.8|       6538|   Marissa Ryan|julianday@example...| 26|East Kimberlychest

**6. Aggregations**

21. Calculate the total sales amount for each product.

22. Find the average age of customers in the customer DataFrame.

23. Calculate the maximum and minimum sales amounts in the sales DataFrame.

24. Group the customer DataFrame by customer_city and count the number of customers in each city.

In [11]:
sales_df.groupBy("product").agg(sum("amount").alias("total_sales")).show()

cust_df.select(avg("age").alias("avg_age")).show()

sales_df.select(max("amount").alias("max_amount"),min("amount").alias("min_amount")).show()

cust_df.groupBy("customer_city").agg(count("customer_id").alias("cust_per_city")).show(5)

+-------+-----------+
|product|total_sales|
+-------+-----------+
| Laptop| 1315091293|
| Mobile| 1317116920|
| Tablet| 1311541968|
|Desktop| 1322736318|
+-------+-----------+

+-------+
|avg_age|
+-------+
|43.6071|
+-------+

+----------+----------+
|max_amount|min_amount|
+----------+----------+
|     99999|      5000|
+----------+----------+

+-------------+-------------+
|customer_city|cust_per_city|
+-------------+-------------+
|   Lake Lucas|            1|
|  Port Monica|            3|
|     Woodstad|            1|
|    Millsview|            2|
|    Dianaland|            1|
+-------------+-------------+
only showing top 5 rows


**7. Sorting**

25. Sort the sales DataFrame by amount in descending order.

26. Sort the customer DataFrame by age in ascending order.

In [12]:
sales_df.select("product","amount").orderBy("amount",ascending=True).show(5)

cust_df.select("customer_id","age").orderBy("age",ascending=True).show(5)

+-------+------+
|product|amount|
+-------+------+
| Tablet|  5000|
| Tablet|  5002|
| Laptop|  5003|
| Tablet|  5004|
| Laptop|  5004|
+-------+------+
only showing top 5 rows
+-----------+---+
|customer_id|age|
+-----------+---+
|        610| 18|
|        722| 18|
|        636| 18|
|        188| 18|
|        710| 18|
+-----------+---+
only showing top 5 rows


**8. Union Operations**

27. Add a new dataset for customers and perform a union operation with the customer
DataFrame.

28. Combine the sales DataFrame with another DataFrame containing additional sales
records.


In [23]:
new_sales_df=spark.read.csv("new_sales.csv")
new_cust_df=spark.read.csv("new_cust.csv")

'''Will give Error as new coloumn added in
sales.csv and customers.csv but not in new_sales.csv
and new_customers.csv'''

sales_df.union(new_sales_df)
cust_df.union(new_cust_df)

'Will give Error as new coloumn added in \nsales.csv but not in new_sales.csv'

**9. Window Functions**

29. Rank the sales records based on the amount column.

30. Add a cumulative sum of amount for each product in the sales DataFrame.

31. Add a column that calculates the difference between each customer's amount and the average amount within their product group.

In [19]:
prod_window=Window.partitionBy("product").orderBy("amount")


sales_df.withColumn("rank" ,rank().over(prod_window)).show(5)

sales_df.withColumn("cum_sum",sum("amount").over(prod_window)).show(5)

avg_amount=avg(col("amount")).over(prod_window)
sales_df.withColumn("diff",col("amount")-avg_amount).show(5)

+--------+-----------+-------+------+----------+------+-----------------+----+
|sales_id|customer_id|product|amount| sale_date|region|discounted_amount|rank|
+--------+-----------+-------+------+----------+------+-----------------+----+
|   76546|       6815| Laptop|  5003|2024-07-26| North|           5503.3|   1|
|   87813|       4663| Laptop|  5004|2025-10-06|  West|5504.400000000001|   2|
|   18555|       6979| Laptop|  5008|2024-12-26|  East|           5508.8|   3|
|   83226|       5123| Laptop|  5013|2025-04-24|  West|           5514.3|   4|
|   44404|       6448| Laptop|  5013|2026-02-14| North|           5514.3|   4|
+--------+-----------+-------+------+----------+------+-----------------+----+
only showing top 5 rows
+--------+-----------+-------+------+----------+------+-----------------+-------+
|sales_id|customer_id|product|amount| sale_date|region|discounted_amount|cum_sum|
+--------+-----------+-------+------+----------+------+-----------------+-------+
|   76546|       68

**10. Partitioning**

32. Write the sales DataFrame to a partitioned Parquet file by region.

33. Partition the customer DataFrame by customer_city and save it as a CSV file.

In [20]:
sales_df.write.partitionBy("region").parquet("content/sales_parquet")

cust_df.write.partitionBy("customer_city").csv("content/cust_csv")

**11. Real-World Scenarios**

34. Calculate the percentage contribution of each product to the total sales.

35. Extract the year from sale_date and group by year to calculate total sales.

36. Identify the most purchased product in each region.

37. Add a column to show the difference between the highest and lowest sales for each product.

38. Write the result of the join between sales and customer to parquet file.

39. Identify products that were sold in the last 6 months.

40. Calculate the average sales amount per customer.

In [21]:
total_sales_overall = sales_df.agg(sum("amount").alias("total_sales")).collect()[0]["total_sales"]
product_sales = sales_df.groupBy("product").agg(sum("amount").alias("product_total_sales"))
product_sales_percentage = product_sales.withColumn(
    "percentage_contribution",
    (col("product_total_sales") / total_sales_overall) * 100
).show(3)



sales_by_year = sales_df.withColumn("sale_year", year("sale_date")) \
    .groupBy("sale_year") \
    .agg(sum("amount").alias("total_sales_yearly")) \
    .orderBy("sale_year").show(3)



prod_region= sales_df.groupBy('region', 'product').agg(count('sales_id').alias('count')) \
.withColumn('ranks', rank().over(Window.partitionBy('region').orderBy(desc('count'))))    \
.filter(col("ranks") == 1).show(3)



sales_df.createOrReplaceTempView("sales")
spark.sql("select product, max(amount) as max_amount,                         \
           min(amount) AS min_amount,                                         \
           max(amount) - min(amount) AS sales_range                           \
           from sales                                                         \
          group by product                                                    \
          order by product").show(3)



cust_df.createOrReplaceTempView("customers")
joined_df_sql = spark.sql("select s.*,c.customer_name,c.email, \
                c.age,c.customer_city,c.customer_age_category  \
                from sales s                                   \
                inner join customers c ON s.customer_id = c.customer_id")
joined_df_sql.write.mode("overwrite").parquet("content/sales_customer_joined.parquet")



spark.sql(" select distinct product                         \
            from sales                                      \
            where sale_date >= date_sub(current_date(), 180)").show(3)



spark.sql(" select customer_id, avg(amount) AS average_sales_amount  \
           from sales                                                \
           group by customer_id                                      \
           order by customer_id").show(3)


+-------+-------------------+-----------------------+
|product|product_total_sales|percentage_contribution|
+-------+-------------------+-----------------------+
| Laptop|         1315091293|     24.970942073993914|
| Mobile|         1317116920|      25.00940466191443|
| Tablet|         1311541968|      24.90354752165482|
+-------+-------------------+-----------------------+
only showing top 3 rows
+---------+------------------+
|sale_year|total_sales_yearly|
+---------+------------------+
|     2024|        2022544981|
|     2025|        2634717081|
|     2026|         609224437|
+---------+------------------+

+------+-------+-----+-----+
|region|product|count|ranks|
+------+-------+-----+-----+
|  East|Desktop| 6379|    1|
| North| Tablet| 6310|    1|
| South| Laptop| 6297|    1|
+------+-------+-----+-----+
only showing top 3 rows
+-------+----------+----------+-----------+
|product|max_amount|min_amount|sales_range|
+-------+----------+----------+-----------+
|Desktop|     99999| 